# Rolling forecast origin schedule

The evaluation schedule is determined from the end of the processed dataset:

- origins are Mondays and forecast the complete Monday-to-Sunday week;
- the last origin is the latest Monday whose full seven-day horizon is observed;
- origins are spaced seven days apart and the schedule retains at most 20 origins, counted backward;
- at each origin, every product-store series with at least 28 active days strictly before the origin is assessed;
- demand-day counts and source-group coverage do not determine the schedule or assessed population.

In [ ]:
from pathlib import Path
import json

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

GROUP_COLS = ["ARTIKEL_ID", "MARKT_ID"]
DEMAND_COL = "ABVERKAUFTE_MENGE_KG"
GROUP_ORDER = ["FCM", "Pseudo"]
PROJECT_ROOT = next(
    (p.resolve() for p in [Path("../.."), Path(".."), Path(".")] if (p / "src").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root containing src/.")
FORECAST_DESIGN_PATH = PROJECT_ROOT / "reports/config/forecast_design.json"

def load_forecast_design(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

forecast_design = load_forecast_design(FORECAST_DESIGN_PATH)

MIN_ACTIVE_DAYS = int(forecast_design.get("MIN_ACTIVE_DAYS", 28))
MIN_FCM_ELIGIBILITY_RATE = 0.80
ORIGIN_SPACING_DAYS = int(forecast_design.get("ORIGIN_SPACING_DAYS", 7))
MAX_ORIGINS = int(forecast_design.get("MAX_ORIGINS", 20))
FORECAST_HORIZON_CALENDAR_DAYS = int(forecast_design.get("FORECAST_HORIZON_DAYS", 7))



configured_data_dir = Path(forecast_design.get("DATA_DIR", "data/processed/transactions"))
DATA_DIR = (
    configured_data_dir if configured_data_dir.is_absolute()
    else PROJECT_ROOT / configured_data_dir
)
parquet_files = sorted(DATA_DIR.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"No parquet files found in {DATA_DIR}")
DATA_GLOB = str(DATA_DIR / "*.parquet")

con = duckdb.connect()
con.execute("PRAGMA threads=4")

## Data basis and audit

Daily rows are built from the configured processed transaction dataset. Active-day counts use `is_active`; closed dates do not contribute to the 28-day pre-origin requirement.

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW source_rows AS
SELECT
    ARTIKEL_ID,
    MARKT_ID,
    CAST(DATE AS DATE) AS period,
    CAST(COALESCE({DEMAND_COL}, 0) AS DOUBLE) AS demand,
    CAST(is_active AS BOOLEAN) AS is_active,
    is_fcm,
    is_pseudo,
    WGR_ID::INTEGER AS category_id,
    CASE WHEN is_fcm THEN 'FCM' WHEN is_pseudo THEN 'Pseudo' END AS sourcing_group
FROM read_parquet('{DATA_GLOB}')
WHERE (is_fcm OR is_pseudo)
  AND WGR_ID IN (890, 900)
""")
con.execute("""
CREATE OR REPLACE TEMP TABLE daily_rows AS
SELECT
    ARTIKEL_ID,
    MARKT_ID,
    period,
    SUM(demand) AS demand,
    BOOL_OR(is_active) AS is_active,
    ANY_VALUE(is_fcm) AS is_fcm,
    ANY_VALUE(is_pseudo) AS is_pseudo,
    ANY_VALUE(category_id) AS category_id,
    ANY_VALUE(sourcing_group) AS sourcing_group
FROM source_rows
GROUP BY ARTIKEL_ID, MARKT_ID, period
""")

row_audit = con.execute("""
WITH series_attributes AS (
    SELECT
        ARTIKEL_ID,
        MARKT_ID,
        COUNT(DISTINCT sourcing_group) AS source_groups,
        COUNT(DISTINCT category_id) AS categories
    FROM source_rows
    GROUP BY ARTIKEL_ID, MARKT_ID
)
SELECT
    COUNT(*) AS rows,
    MIN(period) AS first_date,
    MAX(period) AS last_date,
    COUNT_IF(is_fcm IS NULL) AS null_fcm_flags,
    COUNT_IF(is_pseudo IS NULL) AS null_pseudo_flags,
    COUNT_IF(is_fcm AND is_pseudo) AS overlapping_source_flags,
    COUNT_IF(is_active IS NULL) AS null_active_flags,
    (SELECT COUNT(*) FROM series_attributes WHERE source_groups > 1)
        AS changing_source_series,
    (SELECT COUNT(*) FROM series_attributes WHERE categories > 1)
        AS changing_category_series
FROM source_rows
""").fetchdf()
display(row_audit)

failure_columns = [
    "null_fcm_flags", "null_pseudo_flags", "overlapping_source_flags",
    "null_active_flags", "changing_source_series", "changing_category_series",
]
if row_audit[failure_columns].sum(axis=1).iloc[0]:
    raise RuntimeError("Source/category audit failed; inspect row_audit.")

population = con.execute("""
SELECT
    sourcing_group,
    COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID)) AS series,
    COUNT(DISTINCT ARTIKEL_ID) AS products,
    COUNT(DISTINCT MARKT_ID) AS stores,
    MIN(period) AS first_date,
    MAX(period) AS last_date
FROM daily_rows
GROUP BY sourcing_group
ORDER BY CASE sourcing_group WHEN 'FCM' THEN 1 ELSE 2 END
""").fetchdf()
display(population)


## Origin schedule

The schedule counts backward from the last complete Monday-to-Sunday forecast week. The per-origin assessment rule is applied later by the benchmark and model runners.

In [ ]:
def series_metrics_at(cutoff):
    """Summarize each known series using rows strictly before cutoff."""
    return con.execute("""
    SELECT
        sourcing_group,
        ARTIKEL_ID,
        MARKT_ID,
        COUNT_IF(is_active) AS active_days,
        COUNT_IF(demand > 0) AS demand_days,
        SUM(demand) AS cumulative_demand_kg,
        MIN(period) AS first_period,
        MAX(period) AS last_period
    FROM daily_rows
    WHERE period < ?
    GROUP BY sourcing_group, ARTIKEL_ID, MARKT_ID
    """, [pd.Timestamp(cutoff).date()]).fetchdf()


def group_assessment(metrics):
    rows = []
    for group_name in GROUP_ORDER:
        group = metrics[metrics["sourcing_group"] == group_name]
        assessed = group[group["active_days"] >= MIN_ACTIVE_DAYS]
        rows.append({
            "sourcing_group": group_name,
            "known_series": len(group),
            "assessed_series": len(assessed),
            "assessed_share": len(assessed) / len(group) if len(group) else np.nan,
            "median_active_days": group["active_days"].median(),
            "median_demand_days": group["demand_days"].median(),
        })
    return pd.DataFrame(rows)


In [ ]:
extract_dates = con.execute("""
SELECT MIN(period) AS first_date, MAX(period) AS last_date
FROM daily_rows
""").fetchdf().iloc[0]

latest_test_start = pd.Timestamp(extract_dates["last_date"]) - pd.Timedelta(
    days=FORECAST_HORIZON_CALENDAR_DAYS - 1
)
last_complete_monday = latest_test_start - pd.Timedelta(
    days=latest_test_start.weekday()
)
first_possible_date = pd.Timestamp(extract_dates["first_date"]) + pd.Timedelta(
    days=MIN_ACTIVE_DAYS
)
first_possible_monday = first_possible_date + pd.Timedelta(
    days=(-first_possible_date.weekday()) % 7
)
complete_mondays = pd.date_range(
    first_possible_monday, last_complete_monday, freq="W-MON"
)
forecast_origins = complete_mondays[-MAX_ORIGINS:]
if len(forecast_origins) == 0:
    raise RuntimeError("The dataset has no complete forecast origin.")

first_origin = forecast_origins[0]
first_origin_metrics = series_metrics_at(first_origin)
first_origin_diagnostics = group_assessment(first_origin_metrics)
origin_schedule = pd.DataFrame({"origin": forecast_origins})
origin_schedule["test_end_inclusive"] = (
    origin_schedule["origin"]
    + pd.Timedelta(days=FORECAST_HORIZON_CALENDAR_DAYS - 1)
)

assert len(forecast_origins) <= MAX_ORIGINS
assert (forecast_origins.weekday == 0).all()
assert origin_schedule["test_end_inclusive"].max() <= pd.Timestamp(
    extract_dates["last_date"]
)
print(f"First evaluation origin: {first_origin.date()} ({first_origin.day_name()})")
print(f"Last complete origin: {forecast_origins[-1].date()}")
print(f"Number of weekly origins: {len(forecast_origins)}")
display(origin_schedule)


## First-origin population diagnostic

These diagnostics describe how many known series have at least 28 active pre-origin days at the first retained origin. They do not select or move the origin.

In [ ]:
display(first_origin_diagnostics.style.format({
    "known_series": "{:,.0f}",
    "assessed_series": "{:,.0f}",
    "assessed_share": "{:.1%}",
    "median_active_days": "{:,.1f}",
    "median_demand_days": "{:,.1f}",
}))


## Information boundary and result

The final check verifies that origin design uses only dates available in the extract and writes the shared configuration used by all model runners.

In [ ]:
latest_information_date = con.execute(
    "SELECT MAX(period) FROM daily_rows WHERE period < ?", [first_origin.date()]
).fetchone()[0]
assert pd.Timestamp(latest_information_date) < first_origin
assert first_origin.weekday() == 0

result = pd.DataFrame([{
    "minimum_active_days": MIN_ACTIVE_DAYS,
    "first_origin": first_origin,
    "last_origin": forecast_origins[-1],
    "number_of_origins": len(forecast_origins),
    "origin_spacing_days": ORIGIN_SPACING_DAYS,
    "forecast_horizon_days": FORECAST_HORIZON_CALENDAR_DAYS,
    "latest_information_date": pd.Timestamp(latest_information_date),
}])
forecast_design = {
    "DATA_DIR": forecast_design.get("DATA_DIR", "data/processed/transactions"),
    "MIN_ACTIVE_DAYS": int(MIN_ACTIVE_DAYS),
    "FIRST_ORIGIN": first_origin.strftime("%Y-%m-%d"),
    "ORIGIN_SPACING_DAYS": int(ORIGIN_SPACING_DAYS),
    "FORECAST_HORIZON_DAYS": int(FORECAST_HORIZON_CALENDAR_DAYS),
    "MAX_ORIGINS": int(MAX_ORIGINS),
}

with FORECAST_DESIGN_PATH.open("w", encoding="utf-8") as f:
    json.dump(forecast_design, f, indent=2)
    f.write("\n")
print(f"Forecast design saved to {FORECAST_DESIGN_PATH}")
display(result)
